# Laboratory Exercise 5: File I/O, Error Handling & Object-Oriented Programming
## Instructor's Reference Solution Key
**Duration:** 2 Hours (120 Minutes) | **Total Points:** 100 Points

### Setup: Required Libraries

In [1]:
import json
import os

### Part 1: Robust File I/O & Defensive Exception Handling (15 Points)
#### Task 1.1: Flight Log Writer & Reader (8 Points)
Write formatted waypoint log entries to `flight_log.txt` and read them back safely.

In [3]:
# Task 1.1
log_file = "flight_log.txt"
with open(log_file, "w") as f:
    f.write("WP01,10:15,FL180,250\n")
    f.write("WP02,10:35,FL280,310\n")
    f.write("WP03,11:05,FL350,420\n")

print("--- Reading flight_log.txt ---")
with open(log_file, "r") as f:
    for line in f:
        print(line.strip())

--- Reading flight_log.txt ---
WP01,10:15,FL180,250
WP02,10:35,FL280,310
WP03,11:05,FL350,420


#### Task 1.2: Defensive Exception Handling (7 Points)
Implement `safe_read_altitude(filename)` with `FileNotFoundError`, `ValueError`, and `finally` handling.

In [4]:
# Task 1.2
def safe_read_altitude(filename):
    try:
        with open(filename, "r") as f:
            val = float(f.read().strip())
            return val
    except FileNotFoundError:
        print("[ERROR] Specified flight file not found on disk.")
    except ValueError:
        print("[ERROR] Corrupted altitude record; non-numeric data encountered.")
    finally:
        print("[LOG] File operation attempt concluded.")
    return None

safe_read_altitude("non_existent_alt.txt")

[ERROR] Specified flight file not found on disk.
[LOG] File operation attempt concluded.


### Part 2: JSON Data Interchange & State Serialization (20 Points)
#### Task 2.1: Exporting Flight Dispatch Plan to JSON (10 Points)
Serialize `dispatch_plan` dictionary to `flight_plan.json` with indentation.

In [8]:
# Task 2.1
dispatch_plan = {
    "flight_number": "TG920",
    "origin": "VTBS",
    "destination": "EDDF",
    "aircraft": "Airbus A350-900",
    "cruising_alt_ft": 38000,
    "passengers": 284,
    "fuel_required_kg": 46500.0,
    "alternate_airports": ["EDDM", "EDDL"]
}

json_file = "flight_plan.json"
with open(json_file, "w") as f:
    json.dump(dispatch_plan, f, indent=4)

with open(json_file, "r") as f:
    loaded_plan = json.load(f)

print(f"\nTask 2.1 - Destination: {loaded_plan['destination']}")
print(f"Task 2.1 - Alternates: {loaded_plan['alternate_airports']}")


Task 2.1 - Destination: EDDF
Task 2.1 - Alternates: ['EDDM', 'EDDL']


#### Task 2.2: Updating JSON Records (10 Points)
Append alternate airports, add fuel contingency, and overwrite the JSON file.

In [9]:
# Task 2.2
loaded_plan["alternate_airports"].append("EDDK")
loaded_plan["fuel_required_kg"] += 2500.0

with open(json_file, "w") as f:
    json.dump(loaded_plan, f, indent=4)
print("Task 2.2 - Updated plan successfully written to disk.")

with open(json_file, "r") as f:
    json_loaded_plan = json.load(f)
print(json_loaded_plan)

Task 2.2 - Updated plan successfully written to disk.
{'flight_number': 'TG920', 'origin': 'VTBS', 'destination': 'EDDF', 'aircraft': 'Airbus A350-900', 'cruising_alt_ft': 38000, 'passengers': 284, 'fuel_required_kg': 49000.0, 'alternate_airports': ['EDDM', 'EDDL', 'EDDK']}


### Part 3: OOP Fundamentals — Classes & Instance Variables (20 Points)
#### Task 3.1 & 3.2: Defining the Airport Class & Object Instantiation (20 Points)
Define the `Airport` class, instantiate objects, and verify instance isolation.

In [ ]:
# Task 3.1 & 3.2
class Airport:
    def __init__(self, icao_code, name, elevation_ft, num_runways):
        self.icao_code = icao_code
        self.name = name
        self.elevation_ft = float(elevation_ft)
        self.num_runways = int(num_runways)
        
    def get_info(self):
        return f"{self.name} ({self.icao_code}) | Elevation: {self.elevation_ft:.0f} ft | Runways: {self.num_runways}"

apt1 = Airport("VTBS", "Suvarnabhumi Airport", 5.0, 2)
apt2 = Airport("VTCC", "Chiang Mai International", 1037.0, 1)

print("\nTask 3.2 Objects:")
print(apt1.get_info())
print(apt2.get_info())

### Part 4: Practical Modeling: Aircraft Flight Telemetry Engine (25 Points)
#### Task 4.1: Aircraft Flight Engine Class & Methods
Model aircraft takeoff, cruise, fuel burn, and flight status.

In [ ]:
# Task 4.1
class Aircraft:
    def __init__(self, callsign, model, fuel_kg, cruise_speed_kts):
        self.callsign = callsign
        self.model = model
        self.fuel_kg = float(fuel_kg)
        self.cruise_speed_kts = float(cruise_speed_kts)
        self.altitude_ft = 0.0
        self.is_airborne = False

    def takeoff(self, target_altitude_ft):
        if self.fuel_kg < 1000.0:
            print("[ABORT] Insufficient fuel for takeoff!")
            return False
        self.is_airborne = True
        self.altitude_ft = float(target_altitude_ft)
        self.fuel_kg -= 800.0
        print(f"[TAKEOFF] {self.callsign} ({self.model}) climbed to {self.altitude_ft:.0f} ft. Fuel burn: 800.0 kg.")
        return True

    def fly(self, hours):
        if not self.is_airborne:
            print("[ERROR] Aircraft cannot cruise while on ground.")
            return
        fuel_burn = hours * 2400.0
        if fuel_burn > self.fuel_kg:
            self.fuel_kg = 0.0
            self.altitude_ft = 0.0
            self.is_airborne = False
            print("[EMERGENCY] Fuel exhaustion!")
        else:
            self.fuel_kg -= fuel_burn
            dist = hours * self.cruise_speed_kts
            print(f"[CRUISE] {self.callsign} cruised for {hours} hrs. Distance: {dist:.1f} NM | Fuel burned: {fuel_burn:.1f} kg.")

    def get_status(self):
        return (f"\n--- Aircraft Telemetry Status ---\n"
                f"Callsign: {self.callsign} ({self.model})\n"
                f"Airborne: {self.is_airborne}\n"
                f"Current Altitude: {self.altitude_ft:,.0f} ft\n"
                f"Remaining Fuel: {self.fuel_kg:,.1f} kg")

plane1 = Aircraft("THA911", "Boeing 777-300ER", fuel_kg=12500.0, cruise_speed_kts=480.0)
plane1.takeoff(34000.0)
plane1.fly(3.5)
print(plane1.get_status())

### Part 5: Challenge Task — Flight Dispatch Manager (20 Points)
#### Task 5.1: FlightDispatcher Class with JSON Fleet Persistence
Manage active aircraft fleet, serialize state to JSON, and reload manifest from disk.

In [ ]:
# Task 5.1
class FlightDispatcher:
    def __init__(self):
        self.active_flights = []

    def add_aircraft(self, aircraft_obj):
        self.active_flights.append(aircraft_obj)

    def save_fleet_manifest(self, filepath):
        try:
            records = []
            for ac in self.active_flights:
                records.append({
                    "callsign": ac.callsign,
                    "model": ac.model,
                    "altitude_ft": ac.altitude_ft,
                    "fuel_kg": ac.fuel_kg,
                    "cruise_speed_kts": ac.cruise_speed_kts,
                    "is_airborne": ac.is_airborne
                })
            with open(filepath, "w") as f:
                json.dump(records, f, indent=4)
            print(f"[DISPATCH] Fleet manifest exported to '{filepath}' ({len(records)} aircraft).")
        except IOError as e:
            print(f"[ERROR] Could not write manifest: {e}")

    def load_fleet_manifest(self, filepath):
        try:
            with open(filepath, "r") as f:
                records = json.load(f)
            self.active_flights = []
            for r in records:
                ac = Aircraft(r["callsign"], r["model"], r["fuel_kg"], r["cruise_speed_kts"])
                ac.altitude_ft = r["altitude_ft"]
                ac.is_airborne = r["is_airborne"]
                self.active_flights.append(ac)
            print(f"[DISPATCH] Fleet manifest successfully reloaded from disk.")
        except (FileNotFoundError, IOError) as e:
            print(f"[ERROR] Could not load manifest: {e}")

plane2 = Aircraft("BKP202", "ATR 72-600", fuel_kg=4500.0, cruise_speed_kts=280.0)
dispatcher = FlightDispatcher()
dispatcher.add_aircraft(plane1)
dispatcher.add_aircraft(plane2)

manifest_file = "fleet_manifest.json"
dispatcher.save_fleet_manifest(manifest_file)
dispatcher.load_fleet_manifest(manifest_file)

for idx, ac in enumerate(dispatcher.active_flights, 1):
    print(f"Active Track {idx}: {ac.callsign} | Alt: {ac.altitude_ft:.0f} ft | Fuel: {ac.fuel_kg:.1f} kg")